## Task #10. Hydrogenic orbitals

Plot 2D slices and 3D visualizations of the probability density |ψ|² for an electron in a hydrogenic atom, given *n*, *l*, *m* quantum numbers. Use the provided solutions to the Schrödinger Equation as a recipe (i.e. no need to derive).

In [1]:
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, interact
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401 (registers the '3d' projection)
from numpy import (
    arccos,
    arctan2,
    concatenate,
    cos,
    exp,
    linspace,
    meshgrid,
    pi,
    sin,
    sqrt,
    vstack,
    where,
    zeros_like,
)
from numpy.random import default_rng

C = 2.99792458 * 10**8  # speed of light (m/s)
H = 6.62607015 * 10**-34  # plancks constant (J s)
ME = 9.1093837 * 10**-31  # electron mass (kg)
E_CHARGE = 1.602176634 * 10**-19  # electron charge (C)
EP0 = 8.854187817 * 10**-12  # vacuum permittivity (F/m)

BOHR = (EP0 * H**2) / (pi * ME * E_CHARGE**2)

In [2]:
def Factorial(n):
    if n <= 0:
        return 1
    val = 1
    for x in range(n):
        val *= x + 1
    return val


def Largrange(x, l, n):
    total = zeros_like(x, dtype=float)
    for k in range(n - l):
        num = Factorial(l + n) * (-x) ** k
        dem = Factorial(2 * l + 1 + k) * Factorial(n - l - 1 - k) * Factorial(k)
        if dem == 0:
            continue
        total += num / dem
    return total


def Rcalc(n, l, r):
    rho = (2 * r) / (BOHR * n)
    numSqrt = Factorial(n - l - 1)
    demSqrt = 2 * n * Factorial(n + l)
    sqrtValue = sqrt(numSqrt / demSqrt)
    firstThird = sqrtValue * ((2 / (BOHR * n)) ** 1.5)
    secondThird = (rho**l) * exp(-rho / 2)
    thirdThird = Largrange(rho, l, n)
    return firstThird * secondThird * thirdThird


def AssociatedLegendreP(x, l, m):
    mAbs = abs(m)
    total = zeros_like(x, dtype=float)
    for k in range(mAbs, l + 1):
        if (2 * k - l - mAbs) < 0:
            continue
        num = Factorial(2 * k) * (x ** (2 * k - l - mAbs))
        dem = Factorial(k) * Factorial(l - k) * Factorial(2 * k - l - mAbs)
        total += ((-1) ** (l - k)) * (num / dem)

    Plm = ((-1) ** mAbs) * ((1 - x**2) ** (mAbs / 2)) / (2**l) * total
    if m < 0:
        Plm *= ((-1) ** mAbs) * (Factorial(l - mAbs) / Factorial(l + mAbs))
    return Plm


def OmegaCalc(theta, phi, l, m):
    mAbs = abs(m)
    N = sqrt(((2 * l + 1) / (4 * pi)) * (Factorial(l - mAbs) / Factorial(l + mAbs)))
    P = AssociatedLegendreP(cos(theta), l, mAbs)

    if m < 0:
        return sqrt(2) * N * P * sin(mAbs * phi)
    if m == 0:
        return N * P
    return sqrt(2) * N * P * cos(mAbs * phi)


def ComputeMeshSlice(X, Y, Z, n, l, m):
    r = sqrt(X**2 + Y**2 + Z**2)
    rSafe = where(r == 0, 1e-20, r)
    theta = arccos(Z / rSafe)
    phi = arctan2(Y, X)
    Psi = Rcalc(n, l, r) * OmegaCalc(theta, phi, l, m)
    return Psi**2


def GenerateAllPlanes(n, l, m, gridSize, maxRange):
    axis = linspace(-maxRange, maxRange, gridSize)

    Xxy, Yxy = meshgrid(axis, axis, indexing="ij")
    Xxz, Zxz = meshgrid(axis, axis, indexing="ij")
    Yyz, Zyz = meshgrid(axis, axis, indexing="ij")

    return {
        "xy": ComputeMeshSlice(Xxy, Yxy, zeros_like(Xxy), n, l, m),
        "xz": ComputeMeshSlice(Xxz, zeros_like(Xxz), Zxz, n, l, m),
        "yz": ComputeMeshSlice(zeros_like(Yyz), Yyz, Zyz, n, l, m),
    }


def ProbDensity(points, n, l, m):
    r = sqrt((points**2).sum(axis=1))
    rSafe = where(r == 0, 1e-20, r)
    theta = arccos(points[:, 2] / rSafe)
    phi = arctan2(points[:, 1], points[:, 0])
    Psi = Rcalc(n, l, r) * OmegaCalc(theta, phi, l, m)
    return Psi**2


def SampleOrbitalPoints(n, l, m, nPoints, maxRange, seed=0, maxIter=200):
    rng = default_rng(seed)
    probe = rng.uniform(-maxRange, maxRange, size=(200_000, 3))
    probMax = ProbDensity(probe, n, l, m).max() * 1.2

    keptPoints, keptProbs, total = [], [], 0
    for _ in range(maxIter):
        batch = rng.uniform(-maxRange, maxRange, size=(nPoints, 3))
        prob = ProbDensity(batch, n, l, m)
        probMax = max(probMax, prob.max())
        accept = rng.uniform(0, probMax, size=nPoints) < prob
        keptPoints.append(batch[accept])
        keptProbs.append(prob[accept])
        total += accept.sum()
        if total >= nPoints:
            break

    points = vstack(keptPoints)[:nPoints]
    probs = concatenate(keptProbs)[:nPoints]
    return points, probs

In [3]:
def Orbitals(N, L, M, GridSize=200, MaxRangeBohr=18):
    L = min(L, N - 1)
    M = max(-L, min(L, M))
    maxRange = MaxRangeBohr * BOHR

    probs = GenerateAllPlanes(N, L, M, GridSize, maxRange)
    globalMax = max(probs["xy"].max(), probs["xz"].max(), probs["yz"].max())
    if globalMax == 0:
        globalMax = 1e-20

    extentA = [-maxRange * 1e10, maxRange * 1e10] * 2

    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    cmap = plt.cm.inferno

    ax[0].imshow(
        sqrt(probs["xy"] / globalMax).T,
        extent=extentA,
        origin="lower",
        cmap=cmap,
        vmin=0,
        vmax=1,
    )
    ax[1].imshow(
        sqrt(probs["xz"] / globalMax).T,
        extent=extentA,
        origin="lower",
        cmap=cmap,
        vmin=0,
        vmax=1,
    )
    ax[2].imshow(
        sqrt(probs["yz"] / globalMax).T,
        extent=extentA,
        origin="lower",
        cmap=cmap,
        vmin=0,
        vmax=1,
    )

    ax[0].set_title("XY Plane ($z=0$)")
    ax[1].set_title("XZ Plane ($y=0$)")
    ax[2].set_title("YZ Plane ($x=0$)")

    for a, xLabel, yLabel in zip(ax, ["X", "X", "Y"], ["Y", "Z", "Z"]):
        a.grid(False)
        a.set_xlabel(f"{xLabel} Axis ($\\AA$)")
        a.set_ylabel(f"{yLabel} Axis ($\\AA$)")
        a.set_xlim(-8, 8)
        a.set_ylim(-8, 8)

    fig.colorbar(
        ax[0].images[0],
        ax=ax.tolist(),
        label="Normalized Intensity ($\\sqrt{P}$)",
        fraction=0.02,
        pad=0.04,
    )
    fig.suptitle(f"Hydrogen Orbital Slices (n={N}, l={L}, m={M})", fontsize=14, y=1.02)
    plt.show()

    points, pointProbs = SampleOrbitalPoints(N, L, M, 20000, maxRange)
    pointsA = points * 1e10

    fig3d = plt.figure(figsize=(8, 8))
    ax3d = fig3d.add_subplot(111, projection="3d")
    sc = ax3d.scatter(
        pointsA[:, 0],
        pointsA[:, 1],
        pointsA[:, 2],
        c=pointProbs,
        cmap="inferno",
        alpha=0.15,
        s=3,
        linewidths=0,
    )

    ax3d.set_xlabel("X Axis ($\\AA$)")
    ax3d.set_ylabel("Y Axis ($\\AA$)")
    ax3d.set_zlabel("Z Axis ($\\AA$)")
    ax3d.set_title(f"3D Electron Density Cloud (n={N}, l={L}, m={M})")

    lim = 8
    ax3d.set_xlim(-lim, lim)
    ax3d.set_ylim(-lim, lim)
    ax3d.set_zlim(-lim, lim)

    fig3d.colorbar(sc, ax=ax3d, label="Probability Density", fraction=0.03, pad=0.1)
    plt.show()


interact(
    Orbitals,
    N=IntSlider(value=3, min=1, max=5),
    L=IntSlider(value=2, min=0, max=4),
    M=IntSlider(value=0, min=-4, max=4),
)

interactive(children=(IntSlider(value=3, description='N', max=5, min=1), IntSlider(value=2, description='L', m…

<function __main__.Orbitals(N, L, M, GridSize=200, MaxRangeBohr=18)>